In [11]:
# Hansard Parser
# Author: Joshua Been
# Date: August 5, 2025

# This script is designed to parse multiple PDF files of Parliamentary Hansard reports,
# extract speeches by individual speakers, and compile them into a single,
# clean, and analysis-ready CSV file.

In [12]:
# --- Core Libraries ---
import os
import re
import csv
import html # Used to handle the decoding of HTML entities (e.g., &#x2019; to ').
from datetime import datetime
import json

In [13]:
# --- Dependency Checks ---
# This block checks if several required libraries are installed.
# If not, it uses Python's package manager (pip) to install them.
# This makes the script more portable and easier for others to run.
try:
    import fitz  # PyMuPDF
except ImportError:
    # Use the 'subprocess' module for a more robust way to call pip from within the script.
    import subprocess
    import sys
    print("PyMuPDF not found. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "PyMuPDF"])
    import fitz
    print("Installation complete.")

try:
    import langdetect
except ImportError:
    import subprocess
    import sys
    print("LangDetect not found. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langdetect"])
    import langdetect
    print("Installation complete.")

try:
    import pandas
except ImportError:
    import subprocess
    import sys
    print("Pandas not found. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
    import pandas
    print("Installation complete.")

try:
    from thefuzz import process
except ImportError:
    import subprocess
    import sys
    print("TheFuzz not found. Installing...")
    # It's good to install the optional dependency for faster performance.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "thefuzz[speedup]"])
    from thefuzz import process
    print("Installation complete.")

try:
    from unidecode import unidecode 
except ImportError:
    import subprocess
    import sys
    print("Unidecode not found. Installing...")
    # It's good to install the optional dependency for faster performance.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "unidecode"])
    from unidecode import unidecode 
    print("Installation complete.")

try:
    from tqdm import tqdm # tqdm, used to create a progress bar
except ImportError:
    import subprocess
    import sys
    print("tqdm not found. Installing...")
    # It's good to install the optional dependency for faster performance.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm[tqdm]"])
    from tqdm import tqdm
    print("Installation complete.")

# Enable tqdm for Pandas
tqdm.pandas()

In [14]:
# ==============================================================================
# --- CONFIGURATION SETTINGS ---
# This section contains all the user-configurable variables. Adjust these values
# to change the script's behavior without modifying the core functions below.
# ==============================================================================

from pathlib import Path
# Path to current notebook's directory
notebook_dir = Path.cwd()
# Go one level up, then into Hansards
DOWNLOAD_DIR = notebook_dir.parent / "inputs" /"national_assembly_hansards"
#DOWNLOAD_DIR = notebook_dir.parent / "inputs" / "senate_hansards"

# Define outputs directory (sibling to "scripts")
OUTPUTS_DIR = notebook_dir.parent / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)  # create if it doesn't exist
# The name of the final output CSV file (with outputs directory included)
OUTPUT_CSV_FILE = OUTPUTS_DIR / "national_assembly_parser_outputs.csv"
#OUTPUT_CSV_FILE = OUTPUTS_DIR / "senate_parser_outputs.csv"


In [15]:
# --- Speaker Identification ---
# A list of regular expression patterns to identify lines where a speaker begins.
# This pattern looks for common honorifics and titles that precede a speaker's name.
SPEAKER_PATTERNS = [
    r'^(Hon\.|Sen\.|Dr\.|Prof\.|Mr\.|Mrs\.|Ms\.|The Speaker|The Deputy Speaker|The Temporary Speaker|Hon\. Chairlady|The Temporary Speaker \(Hon\.\)|Hon\. Members)'
]

# --- Content Cleaning ---
# A list of exact, case-sensitive phrases to be completely removed from the text.
# These are typically parenthetical notes about events in the chamber.
EXACT_PHRASES_TO_REMOVE = [
    "(Applause)", "(Laughter)",
    "(Loud consultations)",
    "(Loud Consultations)",
    "(The Quorum Bell was rung)",
    "(The Division Bell was rung)",
    "(Voting in progress)",
    "(Serjeant-at-Arms closed the bars)",
    "(Question put and agreed to)",
    "(Question of the amendment proposed)",
    "(Question proposed)",
    "(The Speaker consulted the Clerks-at-the-Table)",
    "(The Clerk-at-the-Table consulted the Speaker)",
    "Disclaimer: The electronic version of the Official Hansard Report is for information purposes only.",
    "Disclaimer: The electronic version of the Official Hansard Report is for information purposes", # There's actually a line break that normally occurs right here, which is why other versions of this phrase are failing
    "only. A certified version of this Report can be obtained from the Hansard Editor.",
    "A certified version of this Report can be obtained from the Hansard Editor.",
    "Disclaimer: The electronic version of the Senate Hansard Report is for information purposes only.",
    "A certified version of this Report can be obtained from the Hansard Editor, Senate.",
    "A certified version of this Report can be obtained from the Director, Hansard and Audio Services, Senate.",
    "Disclaimer: The electronic version of the Senate Hansard Report is for information purposes",
    "only. A certified version of this Report can be obtained from the Director, Hansard and Audio",
    "Services, Senate.",
    "NATIONAL ASSEMBLY DEBATES",
    "SENATE DEBATES"
]

# A list of more complex patterns to remove using regular expressions.
# These patterns are applied LINE BY LINE to the speech content.
REGEX_PATTERNS_TO_REMOVE = [
    r"\(.*?spoke off the record.*?\)",
    r"\(The Oath of Allegiance was administered to.*?\)",
    r"\(Question carried by.*?\)",
    r"\(The Bill was.*?\)",
    r"\(.*?laid the document on the Table\)"
]

# A new list for patterns that should be removed from the ENTIRE multi-line
# speech content at once, before the line-by-line cleaning begins.
MULTI_LINE_REGEX_PATTERNS_TO_REMOVE = [
    # These patterns remove date/header lines like "December 29, 2020 SENATE DEBATES 31" or " March 16, 2017 PARLIAMENTARY DEBATES 20"
    # even if they are split across multiple lines in the extracted text.
    r'\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}[\s\r\n]+SENATE[\s\r\n]+DEBATES[\s\r\n]+\d+',
    r'\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}[\s\r\n]+PARLIAMENTARY[\s\r\n]+DEBATES[\s\r\n]+\d+'
]

# A list of exact, case-insensitive headings to ignore during parsing.
# These are procedural sections, not speeches, so they should be skipped.
HEADINGS_TO_IGNORE = [
    "PRAYER",
    "PRAYERS", "COMMUNICATION FROM THE CHAIR", "PAPERS LAID", "NOTICE OF MOTION",
    "ORDINARY QUESTIONS", "REQUEST FOR STATEMENTS", "ADMINISTRATION OF OATH",
    "PETITION", "STATEMENTS", "PERSONAL STATEMENT", "MOTION", "MOTIONS", "BILL", "BILLS",
    "DIVISION", "ADJOURNMENT", "PAPER LAID", "PAPER", "PAPERS",
    "DETERMINATION OF QUORUM AT COMMENCEMENT OF SITTING"
]

In [16]:
# ==============================================================================
# --- CORE FUNCTIONS ---
# ==============================================================================

def repair_and_clean_text(text):
    """
    Repairs common encoding errors and cleans text extracted from the PDF.
    This version correctly strips HTML tags before transliterating characters.
    """
    if not text:
        return ""
   
    # --- Step 1: Repair common encoding errors (mojibake) ---
    # This is a preliminary step to fix any obvious character encoding issues
    # that can occur during the initial text extraction from the PDF.
    try:
        repaired_text = text.encode('latin-1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        repaired_text = text
 
    # --- Step 2: Remove any HTML tags FIRST ---
    # This is the most critical step in the cleaning process. The raw text is
    # often wrapped in HTML tags such as <b> and <span>) We must remove these
    # before any other text processing to get to the clean, plain text.
    # Trying to clean the text while the tags are still present (as the
    # original script did) corrupts the tags and prevents them from being removed.
    no_html = re.sub(r'<.*?>', '', repaired_text)
 
    # --- Step 3: Transliterate to plain ASCII using unidecode ---
    # Now that we're working with clean text, we can safely transliterate it
    # to its plainest ASCII representation. This standardizes characters and
    # removes any complex accents or symbols.
    ascii_text = unidecode(no_html)
 
    # Step 4: Handle both real and escaped newlines
    # First, decode escaped sequences (e.g. "\\n" -> "\n")    
    try:
        ascii_text = ascii_text.encode('utf-8').decode('unicode_escape')
    except UnicodeDecodeError: # Ignore warnings when this step fails
        pass
    except DeprecationWarning:
        pass

    # Then replace real newlines and carriage returns
    #ascii_text = ascii_text.replace('\\n', ' ') # original, failed
    #ascii_text = ascii_text.replace('\n', ' ') # failed attempt
    ascii_text = re.sub(r'[\r\n]+', ' ', ascii_text).replace('\\n', ' ') # other failed attempt
 
    # --- Step 5: Normalize whitespace ---
    # Finally, we consolidate any sequences of spaces or tabs into a single
    # space to ensure consistent formatting.
    normalized_space = re.sub(r'[ \t]+', ' ', ascii_text)
    return normalized_space.strip()

In [17]:
def extract_file_metadata(filename, doc):
    """
    Extracts metadata (i.e. date) from the PDF.
    It prioritizes parsing the filename first, then falls back to the document's content.
    We used to also extract Volume Number and Issue Number, but most Hansards don't have those, so not super useful.
    """
    metadata = {"date": None}
    
    # --- Plan A: Parse Date from Filename (e.g., "Hansard - 25th July 2024.pdf") ---
    date_match_name = re.search(r'(\d{1,2})(?:st|nd|rd|th)[\s_]([A-Za-z]+)[\s_](\d{4})', filename, re.IGNORECASE)
    if date_match_name:
        try:
            date_str = f"{date_match_name.group(1)} {date_match_name.group(2)} {date_match_name.group(3)}"
            # Standardize the date format to YYYY-MM-DD for easy sorting and analysis.
            metadata["date"] = datetime.strptime(date_str, "%d %B %Y").strftime("%Y-%m-%d")
        except ValueError:
            # If parsing fails (e.g., "32nd July"), try Plan B.
            pass 

    # --- Plan B: Parse from Document Content ---
    # If the filename didn't contain a usable date, scan the first couple of pages of text.
    if not metadata["date"]:
        text_from_first_pages = ""
        for page_num in range(min(2, len(doc))): # Read first 2 pages, or fewer if doc is short.
            text_from_first_pages += doc[page_num].get_text()
        
        # Try to find common date patterns like "25th July, 2024" or "July 25, 2024".
        date_match1 = re.search(r'\b(\d{1,2})(?:st|nd|rd|th)?\s+([A-Za-z]+),?\s+(\d{4})\b', text_from_first_pages)
        date_match2 = re.search(r'\b([A-Za-z]+)\s+(\d{1,2}),?\s+(\d{4})\b', text_from_first_pages)

        date_str, date_format = None, None
        if date_match1:
            date_str = f"{date_match1.group(1)} {date_match1.group(2)} {date_match1.group(3)}"
            date_format = "%d %B %Y"
        elif date_match2:
            date_str = f"{date_match2.group(1)} {date_match2.group(2)} {date_match2.group(3)}"
            date_format = "%B %d %Y"
        
        if date_str and date_format:
            try:
                metadata["date"] = datetime.strptime(date_str, date_format).strftime("%Y-%m-%d")
            except ValueError:
                metadata["date"] = date_str # Keep the raw string if format is unexpected.

    return metadata

In [18]:
def process_pdf_file(filepath, config):
    """
    The main processing function. It opens a PDF, extracts and cleans the text,
    identifies speakers, assembles their speeches, and returns a list of data records.
    """
    filename = os.path.basename(filepath)
    print(f"--- Processing: {filename} ---")
    try:
        doc = fitz.open(filepath)
    except Exception as e:
        print(f"      Could not open file {filename}. Error: {e}")
        return []
 
    # Extract file metadata at the beginning of the process.
    metadata = extract_file_metadata(filename, doc)
    print(f"      Extracted Metadata: {metadata}")
 
    # --- Step 1: Extract all text blocks from the PDF ---
    # Processing paragraph by paragraph is more robust than using one giant string.
    all_blocks = []
    for page in doc:
        # Extract the page as HTML to preserve paragraph structure.
        page_html = page.get_text("html")
        # Decode HTML entities (e.g., &#x2019; -> ') before further processing.
        decoded_html = html.unescape(page_html)
        # Find all paragraphs within the <p> tags.
        paragraphs = re.findall(r'<p.*?>(.*?)</p>', decoded_html, re.DOTALL)
        for p_html in paragraphs:
            # Clean each paragraph individually using our corrected helper function.
            # This ensures we are working with clean, plain text from the start.
            p_text = repair_and_clean_text(p_html)
            if p_text:
                all_blocks.append({'text': p_text})
    doc.close()
 
    # --- Step 2: First Pass - Identify all speakers and their locations ---
    speakers_with_locations = []
    # Combine all speaker patterns into a single regex for efficiency.
    combined_speaker_pattern = '|'.join(config['speaker_patterns'])
    for i, block in enumerate(all_blocks):
        text = block['text']
        # Ignore blocks that are just procedural headings.
        if text.strip().upper() in config['headings_to_ignore']:
            continue
       
        # A speaker line is assumed to contain a colon.
        if ':' in text:
            potential_speaker = text.split(':', 1)[0].strip()
            # Check if the start of the line matches our speaker patterns.
            if re.search(combined_speaker_pattern, potential_speaker, re.IGNORECASE):
                speakers_with_locations.append({"name": potential_speaker, "block_index": i})
 
    # --- Step 3: Second Pass - Extract content between speakers ---
    extracted_data = []
    for i, current_speaker in enumerate(speakers_with_locations):
        speaker_name = current_speaker["name"]
        start_block_index = current_speaker["block_index"]
       
        # The speech ends where the next speaker begins, or at the end of the document.
        end_block_index = len(all_blocks)
        if i + 1 < len(speakers_with_locations):
            end_block_index = speakers_with_locations[i+1]["block_index"]
 
        # --- Assemble the full speech content ---
        content_accumulator = []
        # The first line of the speech is the text after the colon on the speaker's line.
        first_block_text = all_blocks[start_block_index]['text']
        if ':' in first_block_text:
            content_accumulator.append(first_block_text.split(':', 1)[1].strip())
 
        # Add all subsequent text blocks until the next speaker is found.
        for b_index in range(start_block_index + 1, end_block_index):
            block_text = all_blocks[b_index]['text']
            # Stop if a procedural heading is encountered mid-speech.
            if block_text.strip().upper() in config['headings_to_ignore']:
                break
            content_accumulator.append(block_text)
 
        # Join all parts of the speech, preserving paragraph breaks with a special marker.
        full_content = " [NEWLINE] ".join(filter(None, content_accumulator))
 
        # --- NEW: Consolidated Cleaning Step ---
        # This is the new, centralized location for removing exact phrases. I've
        # moved this logic here to run on the entire, assembled speech content BEFORE
        # it gets split into lines. This allows the script to find multi-line
        # phrases (like the disclaimer) that the old line-by-line method would miss.
        for phrase in config['exact_phrases_to_remove']:
            # Escape any special characters in the phrase (like parentheses).
            pattern = re.escape(phrase)
            # Make the pattern flexible by replacing spaces with a regex command that matches any kind of whitespace. This handles weird spacing from PDFs.
            flexible_pattern = pattern.replace('\\ ', r'\s+')
            # Use re.sub with IGNORECASE to perform a robust, case-insensitive removal.
            full_content = re.sub(flexible_pattern, '', full_content, flags=re.IGNORECASE)
 
        # These patterns specifically target headers that may be split across multiple lines.
        for pattern in MULTI_LINE_REGEX_PATTERNS_TO_REMOVE:
            full_content = re.sub(pattern, '', full_content, flags=re.IGNORECASE)
       
        cleaned_lines = []
        for line in full_content.split(" [NEWLINE] "):
            temp_line = line
            # --- OLD METHOD (DEACTIVATED) ---
            # The original script tried to remove phrases here, one line at a time.
            # I've commented this out because it was ineffective and has been replaced
            # by the more powerful, consolidated method above.
            # for note in config['exact_phrases_to_remove']:
            #     temp_line = temp_line.replace(note, '')
           
            # Remove more complex patterns via regex.
            for pattern in config['regex_patterns_to_remove']:
                temp_line = re.sub(pattern, '', temp_line, flags=re.IGNORECASE)
           
            # Consolidate extra spaces within the line.
            cleaned_line = re.sub(r'\\s+', ' ', temp_line).strip()
            if cleaned_line:
                 cleaned_lines.append(cleaned_line)
 
        final_content = '\n'.join(cleaned_lines)
 
        if not final_content:
            continue
 
        # --- Chunking Logic for Final Output ---
        # An individual speaker's content being too long can cause two downstream problems.
        # We ultimately export as a CSV, and Excel can't handle much more than 30,000 characters per cell.
        # Separately, the analysis notebook uses ML models that have a processing limit of 512 tokens.
        # Tokens and words are a bit different, but a 480 word limit and a 30k character limit should solve both problems.

        MAX_WORDS = 480
        MAX_CHARS = 30000
       
        is_first_chunk = True
        remaining_content = final_content.strip()
       
        while remaining_content:
            words = remaining_content.split()
            if len(words) > MAX_WORDS:
                chunk_words = words[:MAX_WORDS]
                chunk = " ".join(chunk_words)
       
                if len(chunk) > MAX_CHARS:
                    chunk = chunk[:MAX_CHARS]
       
                remaining_content = " ".join(words[MAX_WORDS:])
            else:
                chunk = " ".join(words)
                if len(chunk) > MAX_CHARS:
                    chunk = chunk[:MAX_CHARS]
                remaining_content = ""
 
            source_link_formula = f'=HYPERLINK("{filename}", "Open PDF")'
           
            record = {
                "filename": filename,
                "speaker": speaker_name,
                "content": chunk,
                "source_link": source_link_formula,
                "is_continuation": not is_first_chunk
            }
            if is_first_chunk:
                record.update(metadata)
            else:
                record.update({"date": ""})
 
            extracted_data.append(record)
            is_first_chunk = False
 
    return extracted_data

## Note! is_continuation is not a reference to whether the speaker is the same as the previous block,
## but rather to whether the original block has been broken into several chunks, all of which get is_continuation = True after the first chunk.

In [19]:
def infer_speaker_metadata(record, official_names_list):
    """
    Identifies a speaker's full name by cleaning the name found in the Hansard
    and then fuzzy matching it against a verified list of official names.

    This entire process is currently only applied to records dated 2017 or later.

    Args:
        record (dict): A dictionary representing one row of extracted Hansard data.
                       It must contain a 'speaker' and 'date' key.
        official_names_list (list): A list of strings, where each string is the
                                    full, official name of a parliament member.

    Returns:
        dict: The original record dictionary, now updated with two new keys:
              'matched_name' (the best match found) and 'match_score' (the
              confidence score of that match from 0-100).
    """
    # ==========================================================================
    # --- CONFIGURATION ---
    # These are the main settings you might want to adjust.
    # ==========================================================================

    # The year from which to start matching speakers.
    # Currently we only have speaker data since 2017.
    MATCH_SINCE_YEAR = 2017

    # The confidence threshold for a fuzzy match (from 0 to 100).
    # A higher number means the match must be more precise. 80 is a safe choice.
    MATCH_SCORE_CUTOFF = 80

    # A list of all titles, procedural text, and other "junk" words that
    # should be removed from the speaker's name before attempting a match.
    # This list can be easily added to if new junk phrases are discovered.
    JUNK_WORDS_TO_REMOVE = [
        'The Speaker of the Senate', 'The Speaker', 'The Deputy Speaker',
        'The Temporary Speaker', 'Hon', 'Sen', 'Dr', 'Prof'
    ]

    # ==========================================================================
    # --- CORE LOGIC ---
    # It's generally not necessary to edit below this line.
    # ==========================================================================

    # Step 1: Initialize the new fields in the record. This ensures that even if
    # we don't find a match, the columns will exist in the final CSV.
    record['matched_name'] = ''
    record['match_score'] = ''

    # Step 2: Check if the record is eligible for matching.
    # We only proceed if the record has a date and that date is on or after our target year.
    # This also conveniently skips any "continuation" rows, which have no date.
    if record.get('date') and record['date'][:4].isdigit():
        year = int(record['date'][:4])
        if year >= MATCH_SINCE_YEAR:
            
            speaker_name = record.get('speaker', '').strip()
            
            # Immediately skip records that are for "Hon. Members" or are empty.
            if not speaker_name or 'members' in speaker_name.lower():
                return record

            # Step 3: Intelligent Cleaning. The goal is to isolate the person's actual name.
            
            # A. Prioritize text inside parentheses. If we find "(Sen. Kathuri)",
            # we will start with "Sen. Kathuri" instead of the full line. This is more reliable.
            text_to_clean = speaker_name
            text_in_parentheses = re.search(r'\((.*)\)', speaker_name)
            if text_in_parentheses:
                text_to_clean = text_in_parentheses.group(1)

            # B. Remove all the junk words. We loop through our list and use a regular
            # expression to remove each one from the string we're cleaning.
            cleaned_name = text_to_clean
            for junk in JUNK_WORDS_TO_REMOVE:
                pattern = re.compile(r'\b' + re.escape(junk) + r'\b\.?', re.IGNORECASE)
                cleaned_name = pattern.sub('', cleaned_name)
            
            # C. As a final polish, remove all remaining punctuation and extra whitespace.
            # This leaves us with just the core name, e.g., "Kingi" or "Kathuri".
            final_clean_name = re.sub(r'[^\w\s]', '', cleaned_name).strip()

            # If our cleaning process resulted in an empty string, we can't match it.
            if not final_clean_name:
                return record

            # Step 4: Perform the fuzzy match.
            # We take our clean name and find the single best match in the official list.
            match = process.extractOne(final_clean_name, official_names_list, score_cutoff=MATCH_SCORE_CUTOFF)
            
            # If a confident match was found, add it to our record, and then join additional metadata.
            if match:
                # `match` is a tuple, e.g., ('Amason Jeffah Kingi', 90)
                record['matched_name'] = match[0]
                record['match_score'] = match[1]

    return record


## (2025-09-15) Josh implemented language detection using the langdetect library.
 - Input and Setup: The function receives the full text of each statement/speech and defines parameters such as allowable languages.
 - Chunking: Each statment is split into the component sentences. The language for each sentence is identified. For small *sentences*, which can confuse the language identification process, adjacent sentences of up to 5 words are combined into a single *sentence* for language identification.
 - Contextual Fallback: If a *sentence* is too small or ambiguous to be identified, the function assigns the language of the last successful *sentence*. This assumes a speaker continues in the same language until a clear switch is detected.
 - Percent Calculation: After analyzing all *sentences*, it calculates the final % breakdowns based on total word count.
 - Final Formatting: It creates a dictionary to insert into the output csv.

In [21]:
from langdetect import detect_langs, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
from collections import defaultdict
import json

def analyze_language_distribution(speech_content):
    """
    Analyzes text to determine the distribution of languages. This final version
    uses a 'contextual fallback' method to handle ambiguous, short sentences. If a
    chunk of text is classified as 'unknown', it is assigned the language of the
    previous successfully identified chunk.

    Args:
        speech_content (str): The full text of a single speech.

    Returns:
        str: A JSON string representing the dictionary of language percentages.
    """
    # ==========================================================================
    # --- CONFIGURATION ---
    ALLOWED_LANGUAGES = ['en', 'sw']
    MIN_WORDS_PER_CHUNK = 5
    # The default language to assume at the start of a speech or if no
    # other language can be identified. English is a safe default.
    DEFAULT_LANGUAGE = 'en'
    # ==========================================================================

    # ==========================================================================
    # --- CORE LOGIC ---
    # ==========================================================================
    if not speech_content or not isinstance(speech_content, str):
        return json.dumps({})

    DetectorFactory.seed = 0
    language_word_counts = defaultdict(int)
    total_words = 0
    sentences = re.split(r'[.?!]', speech_content)

    # --- Sentence Chunking Logic ---
    sentence_chunks = []
    current_chunk = ""
    for sentence in sentences:
        current_chunk += sentence + ". "
        if len(current_chunk.split()) >= MIN_WORDS_PER_CHUNK:
            sentence_chunks.append(current_chunk.strip())
            current_chunk = ""
    if current_chunk.strip():
        sentence_chunks.append(current_chunk.strip())
    # --- End of Chunking Logic ---

    # This variable will store the last successfully identified language.
    last_known_language = DEFAULT_LANGUAGE

    for chunk in sentence_chunks:
        if not chunk:
            continue

        words_in_chunk = chunk.split()
        num_words = len(words_in_chunk)
        if num_words == 0:
            continue
        
        total_words += num_words
        detected_language = 'unknown'

        try:
            possible_langs = detect_langs(chunk)
            filtered_langs = [lang for lang in possible_langs if lang.lang in ALLOWED_LANGUAGES]

            if filtered_langs:
                best_match = max(filtered_langs, key=lambda x: x.prob)
                detected_language = best_match.lang
                # We successfully found a language, so we update our tracker.
                last_known_language = detected_language

        except LangDetectException:
            # This chunk is too ambiguous, so it will be handled by the fallback logic.
            pass

        # --- Contextual Fallback Logic ---
        # If the detected language is 'unknown', we assign it the last known language.
        if detected_language == 'unknown':
            final_language = last_known_language
        else:
            final_language = detected_language
        
        language_word_counts[final_language] += num_words

    if total_words == 0:
        return json.dumps({})

    language_percentage = {}
    for lang, count in language_word_counts.items():
        percentage = round((count / total_words), 2)
        # This condition ensures that only languages with a non-zero percentage
        # are included in the final output dictionary.
        if percentage > 0.0:
            language_percentage[lang] = percentage

    return json.dumps(language_percentage)


In [22]:
# ==============================================================================
# --- SCRIPT EXECUTION BLOCK ---
# This is the main part of the script that runs all the steps in order.
# ==============================================================================

if __name__ == "__main__":
    import pandas as pd

    # --- Step 1: Load the Official List of Speaker Names ---
    print("--- Step 1: Loading official speaker names from Google Sheet... ---")
    try:
        sheet_id = '1zVHI2mlEpPHtwvCanyzfEjKuYJEQmHGrLo1jQNa4jrE'
        url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv'
        speaker_df = pd.read_csv(url)
        official_names = speaker_df['Full Name'].dropna().tolist()
        print(f"      Successfully loaded {len(official_names)} names.")
    except Exception as e:
        print(f"      Could not load speaker names. Error: {e}")
        official_names = []

    # --- Step 2: Find and Process All PDF Files ---
    print(f"--- Step 2: Looking for PDF files in '{DOWNLOAD_DIR}'... ---")
    all_speaker_data = []
    config = {
        'speaker_patterns': SPEAKER_PATTERNS,
        'exact_phrases_to_remove': EXACT_PHRASES_TO_REMOVE,
        'regex_patterns_to_remove': REGEX_PATTERNS_TO_REMOVE,
        'headings_to_ignore': HEADINGS_TO_IGNORE
    }
    try:
        files_in_dir = sorted(os.listdir(DOWNLOAD_DIR))
        print(f"      Found {len(files_in_dir)} items in the directory.")
        for filename in files_in_dir:
            if filename.lower().endswith('.pdf'):
                filepath = os.path.join(DOWNLOAD_DIR, filename)
                all_speaker_data.extend(process_pdf_file(filepath, config))
    
    except FileNotFoundError:
        print(f"      Error: The directory '{DOWNLOAD_DIR}' was not found.")

    # --- Step 3: Post-Processing - Match Speaker Names ---
    if all_speaker_data and official_names:
        print("--- Step 3: Matching speaker names against the official list... ---")
        all_speaker_data = [
            infer_speaker_metadata(record, official_names) 
            for record in tqdm(all_speaker_data, desc="Matching speakers", unit="record")
        ]
        print("      Speaker matching complete.")

    # --- Step 4: Post-Processing - Analyze Language Distribution ---
    if all_speaker_data:
        print("--- Step 4: Analyzing language distribution for each speech... ---")
        # We loop through each record, get the content, and add the language analysis results.
        for record in tqdm(all_speaker_data, desc="Analyzing speeches", unit="speech"):
            speech_text = record.get('content', '')
            # We create a new key, 'language_distribution', and store the JSON results from our new function.
            record['language_distribution'] = analyze_language_distribution(speech_text)
        print("      Language analysis complete.")
    
    # --- Step 5: Write All Data to a Final CSV File ---
    if all_speaker_data:
        print(f"--- Step 5: Writing {len(all_speaker_data)} rows to '{OUTPUT_CSV_FILE}'... ---")
        
        # Define the final headers for the CSV file, including our new column.
        fieldnames = [
            'filename', 'date', 'speaker', 
            'matched_name', 'match_score', 'language_distribution', 'content', 
            'source_link', 'is_continuation'
        ]
        
        with open(OUTPUT_CSV_FILE, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_speaker_data)
        
        print(f"Successfully created consolidated CSV file: '{OUTPUT_CSV_FILE}'")
    else:
        print("No speaker data was extracted. The script has finished with no output file created.")


--- Step 1: Loading official speaker names from Google Sheet... ---
      Successfully loaded 763 names.
--- Step 2: Looking for PDF files in 'C:\Users\USER\Dropbox\Research\Ongoing Research\EAC Stuff\Kenyan Parliamentary Records\inputs\national_assembly_hansards'... ---
      Found 1314 items in the directory.
--- Processing: 14.03.2017_HR.pdf ---
      Extracted Metadata: {'date': '2017-03-14'}
--- Processing: 15.03.2017_HR.pdf ---
      Extracted Metadata: {'date': '2017-03-15'}
--- Processing: 16.03.2017_HR.pdf ---
      Extracted Metadata: {'date': '2017-03-16'}
--- Processing: 27.05_2015.pdf ---
      Extracted Metadata: {'date': '2015-05-27'}
--- Processing: HANSARD REPORT - THURSDAY, 13TH MAY 2021(A).pdf ---
      Extracted Metadata: {'date': '2021-05-13'}
--- Processing: HANSARD REPORT - THURSDAY, 13TH MAY 2021(P).pdf ---
      Extracted Metadata: {'date': '2021-05-13'}
--- Processing: HANSARD REPORT - WEDNESDAY, 19TH MAY 2021(A) (SPECIAL SITTING).pdf ---
      Extracted Metad

C:\Users\USER\AppData\Local\Temp\ipykernel_46628\2207708933.py:38: DeprecationWarning: invalid escape sequence '\ '
  ascii_text = ascii_text.encode('utf-8').decode('unicode_escape')


--- Processing: Hansard Report - Thursday, 28th November 2024 (P).pdf ---
      Extracted Metadata: {'date': '2024-11-28'}
--- Processing: Hansard Report - Thursday, 28th September 2023 (P)_0.pdf ---
      Extracted Metadata: {'date': '2023-09-28'}
--- Processing: Hansard Report - Thursday, 29th April 2021 (A)_0.pdf ---
      Extracted Metadata: {'date': '2021-04-29'}
--- Processing: Hansard Report - Thursday, 29th April 2021 (P) - Special Sitting.pdf ---
      Extracted Metadata: {'date': '2021-04-29'}
--- Processing: Hansard Report - Thursday, 29th February 2024 (P).pdf ---
      Extracted Metadata: {'date': '2024-02-29'}
--- Processing: Hansard Report - Thursday, 29th June 2023 (P).pdf ---
      Extracted Metadata: {'date': '2023-06-29'}
--- Processing: Hansard Report - Thursday, 29th May 2025 (P).pdf ---
      Extracted Metadata: {'date': '2025-05-29'}
--- Processing: Hansard Report - Thursday, 29th November, 2018(A).pdf ---
      Extracted Metadata: {'date': '2018-11-29'}
--- Proc

C:\Users\USER\AppData\Local\Temp\ipykernel_46628\2207708933.py:38: DeprecationWarning: invalid escape sequence '\o'
  ascii_text = ascii_text.encode('utf-8').decode('unicode_escape')


--- Processing: Hansard_Report_-_Tuesday_14th_February_2017.pdf ---
      Extracted Metadata: {'date': '2017-02-14'}
--- Processing: Hansard_Report_-_Tuesday_14th_June_2016P.pdf ---
      Extracted Metadata: {'date': '2016-06-14'}
--- Processing: Hansard_Report_-_Tuesday_15th_March_2016P.pdf ---
      Extracted Metadata: {'date': '2016-03-15'}
--- Processing: Hansard_Report_-_Tuesday_15th_November_2016P.pdf ---
      Extracted Metadata: {'date': '2016-11-15'}
--- Processing: Hansard_Report_-_Tuesday_16th_August_2016P.pdf ---
      Extracted Metadata: {'date': '2016-08-16'}
--- Processing: Hansard_Report_-_Tuesday_16th_February_2016P.pdf ---
      Extracted Metadata: {'date': '2016-02-16'}
--- Processing: Hansard_Report_-_Tuesday_16th_June_2015P.pdf ---
      Extracted Metadata: {'date': '2015-06-16'}
--- Processing: Hansard_Report_-_Tuesday_16th_May_2017_P.pdf ---
      Extracted Metadata: {'date': '2017-05-16'}
--- Processing: Hansard_Report_-_Tuesday_16th_May_2017_P_1.pdf ---
      E

...

--- Processing: Hansard_Report_-_Wednesday_22nd_March_2017_P-1.pdf ---
      Extracted Metadata: {'date': '2017-03-22'}
--- Processing: Hansard_Report_-_Wednesday_22nd_March_2017_P-1_1.pdf ---
      Extracted Metadata: {'date': '2017-03-22'}
--- Processing: Hansard_Report_-_Wednesday_23rd_March_2016A.pdf ---
      Extracted Metadata: {'date': '2016-03-23'}
--- Processing: Hansard_Report_-_Wednesday_23rd_March_2016P.pdf ---
      Extracted Metadata: {'date': '2016-03-23'}
--- Processing: Hansard_Report_-_Wednesday_23rd_November_2016A.pdf ---
      Extracted Metadata: {'date': '2016-11-23'}
--- Processing: Hansard_Report_-_Wednesday_23rd_November_2016A_1.pdf ---
      Extracted Metadata: {'date': '2016-11-23'}
--- Processing: Hansard_Report_-_Wednesday_23rd_November_2016P.pdf ---
      Extracted Metadata: {'date': '2016-11-23'}
--- Processing: Hansard_Report_-_Wednesday_24th_April_2013_Afternoon.pdf ---
      Extracted Metadata: {'date': '2013-04-24'}
--- Processing: Hansard_Report_-_Wed

Matching speakers: 100%|██████████████████████████████████████████████████| 220135/220135 [04:34<00:00, 801.67record/s]


      Speaker matching complete.
--- Step 4: Analyzing language distribution for each speech... ---


Analyzing speeches: 100%|██████████████████████████████████████████████████| 220135/220135 [52:01<00:00, 70.52speech/s]


      Language analysis complete.
--- Step 5: Writing 220135 rows to 'C:\Users\USER\Dropbox\Research\Ongoing Research\EAC Stuff\Kenyan Parliamentary Records\outputs\national_assembly_parser_outputs.csv'... ---
Successfully created consolidated CSV file: 'C:\Users\USER\Dropbox\Research\Ongoing Research\EAC Stuff\Kenyan Parliamentary Records\outputs\national_assembly_parser_outputs.csv'


In [23]:
# Test cell to search the outputs

# The outputs don't get written as a df, instead they are a JSON-like list of tuples, which makes searching them a bit harder
all_speaker_data

# Way to search for a specific string 
search_string = "Disclaimer: The electronic version of the Official Hansard Report is for information purposes"
#search_string = "(The Quorum Bell was rung)"

matches = [entry for entry in all_speaker_data if search_string in entry['content']]
matches

[{'filename': 'Hansard Report - Thursday, 16th January 2025 (A).pdf',
  'speaker': 'Hon. Speaker',
  'content': "I take this opportunity to welcome you to this Special Sitting which has been convened pursuant to the provisions of Standing Order 29(3) relating to the procedure for convening Special Sittings of the House during recess. As you will recall, the House proceeded on its Long Recess on 6th December 2024 in line with the approved Calendar of Regular Sittings 2024. With this in mind, I wish to report to the House that on 3rd January 2025, I received a request from the Leader of the Majority Party that I appoint today, Thursday, 16th January 2025, as a day for Special Sittings of the National Assembly. Hon. Members, I duly considered the business proposed to be transacted and acceded to the request. Consequently, and in keeping with the requirements of Standing Order 29, I gazetted today's Special Sittings commencing at 10.00 a.m. for the Morning Sitting and at 2.30 p.m. for the 